# Build the Slideseek index

This notebook is the **offline half** of Slideseek. It turns lecture videos into a search
index on a GPU, once, and publishes the index so the CPU-only demo can serve it.

```
videos  ->  slide segmentation  ->  OCR  ->  speech recognition  ->  moments  ->  embeddings  ->  index
```

**Before running**

- Kaggle: *Settings → Accelerator → GPU T4*, and *Internet → On*. Colab: *Runtime → Change runtime type → T4 GPU*.
- Only index lectures whose licence allows sharing derived transcripts (for example MIT
  OpenCourseWare, CC BY-NC-SA). Recordings from your own university stay on your own
  machine: use `slideseek index` locally for those, not this notebook.

With no lectures configured, the notebook builds two short synthetic demo lectures instead,
so it can be run end to end as a smoke test before any real data is involved.

## 1. Setup

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

PY = sys.executable  # run every command with the notebook's own interpreter

if shutil.which("nvidia-smi"):
    print(
        subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True,
        ).stdout.strip()
    )
else:
    print(
        "No GPU found. Indexing will run on CPU, which is fine for the demo and slow for real lectures."
    )

if shutil.which("apt-get"):
    # ffmpeg decodes the videos; espeak-ng is only needed to narrate the synthetic demo lectures.
    !apt-get -qq install -y ffmpeg espeak-ng > /dev/null

REPO = "https://github.com/lakshyasakhuja/slideseek.git"
if not Path("pyproject.toml").exists():
    if not Path("slideseek").exists():
        !git clone -q {REPO}
    os.chdir("slideseek")
print("working in", Path.cwd())

!{PY} -m pip install -q -e ".[index]"

faster-whisper runs on CTranslate2, which needs cuBLAS and cuDNN at load time. On Kaggle and
Colab those libraries ship inside pip packages rather than on the system path, so the path
is added here. Commands started with `!` inherit it.

In [ ]:
import importlib.util

lib_dirs = []
for package in ("nvidia.cublas", "nvidia.cudnn"):
    try:
        spec = importlib.util.find_spec(package)
    except ModuleNotFoundError:  # no nvidia packages at all: a CPU-only machine
        spec = None
    if spec and spec.submodule_search_locations:
        lib_dirs.append(str(Path(next(iter(spec.submodule_search_locations))) / "lib"))
if lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join([*lib_dirs, os.environ.get("LD_LIBRARY_PATH", "")])

!{PY} -c "import ctranslate2; print('CUDA devices visible to the speech model:', ctranslate2.get_cuda_device_count())"

## 2. Lectures

Each lecture is one video file named `<lecture_id>.mp4`, with a matching row in
`config/lectures.csv` giving its title, course, source page, licence and a link template
such as `https://www.youtube.com/watch?v=VIDEO_ID&t={seconds}s`.

Provide the videos in one of two ways:

- **A Kaggle dataset** you have attached to this notebook: set `KAGGLE_INPUT` to its path.
- **Official download links**: fill in `VIDEO_URLS`. For MIT OpenCourseWare use the
  "Download video" link on each lecture page.

In [ ]:
KAGGLE_INPUT = None  # e.g. "/kaggle/input/my-ocw-lectures"
VIDEO_URLS = {
    # "lec01": "https://.../lec01.mp4",
}

videos = Path("videos")
videos.mkdir(exist_ok=True)
if KAGGLE_INPUT:
    for source in sorted(Path(KAGGLE_INPUT).iterdir()):
        if source.suffix.lower() in {".mp4", ".mkv", ".mov", ".webm", ".m4v"}:
            target = videos / source.name
            if not target.exists():
                target.symlink_to(source)
for lecture_id, url in VIDEO_URLS.items():
    target = videos / f"{lecture_id}.mp4"
    if not target.exists():
        print("downloading", lecture_id)
        subprocess.run(["wget", "-q", "-O", str(target), url], check=True)

USE_DEMO = not any(videos.iterdir())
if USE_DEMO:
    print("No lectures configured: generating the synthetic demo lectures.")
    !{PY} -m slideseek demo demo
    VIDEOS, CATALOG, OUT, WORK = "demo/videos", "demo/lectures.csv", "demo/index", "demo/work"
else:
    VIDEOS, CATALOG, OUT, WORK = "videos", "config/lectures.csv", "index", "work"

for path in sorted(Path(VIDEOS).iterdir()):
    print(f"{path.name:30s} {path.stat().st_size / 1e6:8.1f} MB")

## 3. One-minute check

Transcribe the first minute of the first lecture before committing to the full run. If the
speech model cannot load on the GPU, this is where it fails, in seconds rather than after
the slide stages.

In [ ]:
first_video = sorted(Path(VIDEOS).iterdir())[0]
check = f'''
import time
from slideseek import asr, ingest
clip = ingest.extract_audio("{first_video}", "one_minute.wav", start=0, duration=60)
started = time.time()
model = asr.load_model()
segments = asr.transcribe(clip, model=model)
print("device and precision:", asr.resolve_device())
print(f"transcribed 60 s in {{time.time() - started:.1f}} s, model load included")
print(" ".join(s.text for s in segments)[:400])
'''
_ = subprocess.run([PY, "-c", check], check=True)

## 4. Build the index

Every stage writes its result to `work/<lecture_id>/` before the next begins, and a stage is
skipped when its inputs have not changed. If the session is interrupted, run this cell
again: it resumes where it stopped. Adding a lecture later processes only that lecture.

In [ ]:
!{PY} -m slideseek index {VIDEOS} --catalog {CATALOG} --out {OUT} --work {WORK}

## 5. Look at what was built

In [ ]:
!{PY} -m slideseek info --index {OUT}

A contact sheet of the detected slides for one lecture. A missed slide change shows up as two
different slides sharing one tile's time range; a spurious one as the same slide twice. If a
webcam overlay or progress bar is causing false cuts, add a mask for that lecture in
`config/masks.yaml` and re-run the index cell.

In [ ]:
from IPython.display import Image, display

!{PY} -m slideseek slides {first_video} --sheet contact_sheet.jpg | tail -n 1
display(Image("contact_sheet.jpg"))

In [ ]:
QUERIES = [
    "why do we hold out data we never train on",
    "the plot where one curve goes down and the other goes back up",
]
for query in QUERIES:
    print("=" * 100)
    print(query)
    !{PY} -m slideseek search "{query}" --index {OUT} --top-k 3

## 6. Throughput

Seconds of processing per hour of video, by stage, on this machine. This is the indexing
cost figure reported in the README, alongside the GPU model printed in the first cell.

In [ ]:
import json

manifest = json.loads((Path(OUT) / "manifest.json").read_text())
hours = manifest["total_duration_s"] / 3600
totals = {}
for stages in manifest["stage_seconds"].values():
    for stage, seconds in stages.items():
        totals[stage] = totals.get(stage, 0.0) + seconds
if totals:
    print(f"{hours:.2f} hours of video, {manifest['n_moments']} moments\n")
    for stage, seconds in totals.items():
        print(f"{stage:10s} {seconds:8.0f} s   {seconds / hours:8.0f} s per hour of video")
    print(
        f"{'total':10s} {sum(totals.values()):8.0f} s   {sum(totals.values()) / hours:8.0f} s per hour of video"
    )
else:
    print("Every stage came from the cache, so there are no timings from this run.")

## 7. Publish the index

The index goes to a Hugging Face **dataset** repository, which the demo downloads at
start-up. A dataset card naming every lecture, its source and its licence is generated from
the index and uploaded with it.

Add a write token as a Kaggle secret named `HF_TOKEN` (*Add-ons → Secrets*), or set the
`HF_TOKEN` environment variable on Colab, then set `HF_REPO`.

In [ ]:
HF_REPO = ""  # e.g. "your-username/slideseek-index"

if not HF_REPO:
    print("HF_REPO is not set: nothing published.")
elif USE_DEMO:
    print("This is the synthetic demo index: not publishing it.")
else:
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except ImportError:
        pass  # not on Kaggle: HF_TOKEN must already be in the environment
    !{PY} -m slideseek push {HF_REPO} --index {OUT}

## 8. Keep the cache

`work/` holds every intermediate result, including the transcripts, which are the expensive
part. On Kaggle, *Save Version* keeps `/kaggle/working`; attach that output to a later session
and copy `work/` back to resume or to add lectures without redoing the others. The audio
files are the bulk of it and can be deleted once indexing is finished, unless you plan to
run `eval/eval_asr.py --compare-jargon`, which re-transcribes excerpts from them.